# 12.2 抛物型偏微分方程

热方程 $u_t=\kappa u_{xx}$ 描述扩散和平滑化。显式格式直观但受稳定性限制，隐式格式需要求解线性系统但允许更大的时间步。

In [ ]:
import math
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next(parent for parent in pathlib.Path.cwd().parents if (parent / 'src').exists())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    solve_heat_1d_crank_nicolson,
    solve_heat_1d_ftcs,
    solve_heat_1d_implicit_euler,
)

## 制造解和扩散数

对齐次 Dirichlet 边界，$u(x,t)=e^{-\pi^2t}\sin(\pi x)$ 是 $\kappa=1$ 的解析解。显式 FTCS 的扩散数 $r=\kappa\Delta t/\Delta x^2$ 需要满足稳定性限制。

In [ ]:
points = 51
x = np.linspace(0.0, 1.0, points)
dx = x[1] - x[0]
initial = np.sin(math.pi * x)

ftcs = solve_heat_1d_ftcs(initial, diffusivity=1.0, dx=dx, dt=0.4 * dx * dx, steps=40)
exact_ftcs = np.exp(-math.pi * math.pi * ftcs.times[-1]) * np.sin(math.pi * x)
ftcs.diffusion_number, np.linalg.norm(ftcs.values[-1] - exact_ftcs, ord=np.inf)

## 隐式 Euler

隐式 Euler 每步求解三对角线性系统，数值耗散更强，但可以使用超过显式稳定限制的大步长。

In [ ]:
implicit = solve_heat_1d_implicit_euler(initial, diffusivity=1.0, dx=dx, dt=2.0 * dx * dx, steps=10)
exact_implicit = np.exp(-math.pi * math.pi * implicit.times[-1]) * np.sin(math.pi * x)
implicit.diffusion_number, float(np.max(np.abs(implicit.values[-1]))), np.linalg.norm(implicit.values[-1] - exact_implicit, ord=np.inf)

## Crank-Nicolson

Crank-Nicolson 对时间做梯形平均，通常比隐式 Euler 更准确，但仍需要求解线性系统。

In [ ]:
cn = solve_heat_1d_crank_nicolson(initial, diffusivity=1.0, dx=dx, dt=0.8 * dx, steps=5)
exact_cn = np.exp(-math.pi * math.pi * cn.times[-1]) * np.sin(math.pi * x)
cn.diffusion_number, np.linalg.norm(cn.values[-1] - exact_cn, ord=np.inf)

## 稳定性对比

FTCS 在过大扩散数下可能出现振荡和增长；隐式方法通常不会出现这种显式稳定性限制。

In [ ]:
unstable = solve_heat_1d_ftcs(initial, diffusivity=1.0, dx=dx, dt=0.8 * dx * dx, steps=20)
{
    'stable_ftcs_max': float(np.max(np.abs(ftcs.values[-1]))),
    'large_r_ftcs_max': float(np.max(np.abs(unstable.values[-1]))),
    'large_r': unstable.diffusion_number,
}

## 小结

* FTCS 简单直接，但扩散数过大时会失稳。
* 隐式 Euler 稳定性更强，但数值耗散明显。
* Crank-Nicolson 通过时间中心化提高精度，是热方程常用的基准隐式格式。